# MNIST handwritten-digit classifier

This Google Colab notebook downloads the **MNIST** dataset, prepares separate training and test sets, trains a simple linear classifier, and reports the **test classification error**.

- **Dataset:** 60,000 training images and 10,000 test images, each 28 × 28 pixels (official MNIST split).
- **Preprocessing:** flatten each image to 784 features and normalize pixel values to the range [0, 1].
- **Model:** stochastic-gradient-descent (SGD) linear classifier using logistic loss.
- **Evaluation:** held-out test accuracy, test error rate, and number of incorrect predictions.

**Run in Colab:** Choose **Runtime → Run all**. The first run downloads approximately 11 MB. A CPU runtime is sufficient.

## 1. Import dependencies

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

## 2. Download MNIST

Download the compressed dataset from the TensorFlow-hosted MNIST mirror. The `.npz` archive contains the official training and test partitions.

In [ ]:
DATA_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
DATA_FILE = Path("mnist.npz")

if not DATA_FILE.exists():
    print("Downloading MNIST...")
    urlretrieve(DATA_URL, DATA_FILE)
else:
    print("Using cached MNIST dataset.")

with np.load(DATA_FILE, allow_pickle=False) as data:
    train_images = data["x_train"]
    train_labels = data["y_train"]
    test_images = data["x_test"]
    test_labels = data["y_test"]

print(f"Raw training images: {train_images.shape}")
print(f"Raw test images:     {test_images.shape}")

## 3. Prepare training and test datasets

Use the standard MNIST split to avoid mixing test examples into training. Scale pixel intensities to [0, 1] and flatten the images for scikit-learn.

In [ ]:
X_train = train_images.reshape(-1, 28 * 28).astype(np.float32) / 255.0
X_test = test_images.reshape(-1, 28 * 28).astype(np.float32) / 255.0
y_train = train_labels.astype(np.int64)
y_test = test_labels.astype(np.int64)

assert X_train.shape == (60_000, 784)
assert X_test.shape == (10_000, 784)

print(f"Training: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Test:     {X_test.shape[0]:,} samples, {X_test.shape[1]} features")

## 4. Train a simple classifier

Fit a linear classifier with logistic loss using stochastic gradient descent. **Only the training set is used for fitting.**

In [ ]:
classifier = SGDClassifier(
    loss="log_loss",
    alpha=1e-4,
    max_iter=30,
    tol=1e-3,
    random_state=RANDOM_STATE,
)
classifier.fit(X_train, y_train)
print("Model training complete.")

## 5. Report test-set classification error

The **test error rate** is the fraction of test digits assigned the wrong label: `incorrect / total = 1 − accuracy`. Lower is better.

In [ ]:
test_predictions = classifier.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
incorrect = int(np.count_nonzero(test_predictions != y_test))
test_error = incorrect / len(y_test)

print(f"Test accuracy:       {test_accuracy:.4f} ({test_accuracy:.2%})")
print(f"Test error rate:     {test_error:.4f} ({test_error:.2%})")
print(f"Incorrect predictions: {incorrect:,} / {len(y_test):,}")

**Interpretation:** The printed error is measured on the 10,000 images held out for testing. Results may differ if the model or package versions are changed. No MNIST test examples were used for training.